# 06 — Experimento 2: resultados

Compara, sobre las mismas ventanas y con tu misma referencia, las respuestas de cada modelo de lenguaje en las 4 variantes de la prueba (N0 y N1, con señales en filas y con reportes en filas). **No necesita GPU:** lee las ejecuciones guardadas por `05_experimento_2`.

**Criterio (el de la memoria):** anomalía = 1; normal y anomalía mitigada = 0; las dudosas quedan fuera del cálculo principal y se cuentan aparte de las dos formas. Una ventana sin veredicto (JSON no válido o respuesta cortada) cuenta como fallo.

**Comparación principal, fijada antes de ejecutar la prueba (12/10/2026):** ¿acierta Qwen3.5-9B más que Qwen2.5-7B en N1 con señales en filas, sobre las 40 ventanas de prueba? Prueba exacta de McNemar, sin las dudosas. El resto de modelos y variantes se presentan como resultados descriptivos.

| Sección | Qué hace |
|---|---|
| 1–3 | Entorno, parámetros y carga de las ejecuciones |
| 4 | Resultados: comparación principal, métricas, grupos, N0 frente a N1, pares de modelos, dudosas y tabla por ventana |
| 5 | Revisión: una ventana (gráfica y veredictos) y el informe HTML con las respuestas (privado) |
| 6 | Guardado de las cifras en `resultados/experimento_2/` |

**Confidencial:** el notebook solo muestra cifras, veredictos y nombres cortos. Las respuestas del modelo solo aparecen en el informe HTML, que se guarda en `privado/`.

## 1. Entorno

En Colab, monta Google Drive y clona el repositorio para tener acceso a `src/`. En local, busca la carpeta `src/` subiendo desde el directorio actual.

In [ ]:
# --- Entorno: localiza src/ y, en Colab, monta Google Drive ---
import os
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules
URL_REPOSITORIO = "https://github.com/Williams-MP/TFM_VIU_2026.git"
RUTA_REPO_COLAB = "/content/TFM_VIU_2026"

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(RUTA_REPO_COLAB):
        # Mientras el repositorio sea privado hace falta un token de GitHub.
        # Si es público, basta con dejar el campo vacío.
        from getpass import getpass
        token = getpass("Token de GitHub (vacío si el repositorio es público): ")
        url = URL_REPOSITORIO.replace("https://", f"https://{token}@") if token else URL_REPOSITORIO
        subprocess.run(["git", "clone", "-q", url, RUTA_REPO_COLAB], check=True)
    ruta_src = f"{RUTA_REPO_COLAB}/src"
else:
    # Ejecución local: se sube por las carpetas hasta encontrar src/.
    directorio = os.getcwd()
    while not os.path.isdir(os.path.join(directorio, "src")):
        padre = os.path.dirname(directorio)
        if padre == directorio:
            raise FileNotFoundError("No se encuentra la carpeta src/ del repositorio.")
        directorio = padre
    ruta_src = os.path.join(directorio, "src")

if ruta_src not in sys.path:
    sys.path.append(ruta_src)

import json
from datetime import datetime

import pandas as pd

from config import (RUTA_PREPARADOS, IDX_CONTADORES, IDX_GAP, NOMBRES_CONTADORES,
                    RUTA_EXPERIMENTO_2_PRIVADO, RUTA_EXPERIMENTO_2)
from llm_lab import exp2
from llm_lab.prompts import leer_correspondencia
from llm_lab.ventanas import FuenteVentanas

RUTA_ENTRADAS = f"{RUTA_EXPERIMENTO_2_PRIVADO}/entradas"
RUTA_EJECUCIONES_37 = f"{RUTA_EXPERIMENTO_2_PRIVADO}/ejecuciones_37"
RUTA_EJECUCIONES = f"{RUTA_EXPERIMENTO_2_PRIVADO}/ejecuciones"
RUTA_INFORMES = f"{RUTA_EXPERIMENTO_2_PRIVADO}/informes"     # HTML con las respuestas del modelo (privado)

fuente = FuenteVentanas(RUTA_PREPARADOS, IDX_CONTADORES, IDX_GAP)

## 2. Parámetros

In [ ]:
# --- Parámetros (EDITABLE) ---
CONJUNTO = "test"     # "test" (40 ventanas de prueba) o "dev" (29 de desarrollo)
EJECUCIONES = None    # None = la más reciente de cada modelo en privado/experimento_2/ejecuciones; o {modelo: base}

# Qwen2.5-7B: las ejecuciones de la memoria
REFERENCIA_QWEN25 = {"dev": f"{RUTA_EJECUCIONES_37}/fase1_20261008_1046.csv",
                     "test": f"{RUTA_EJECUCIONES_37}/fase2_20261008_1127.csv"}

# Comparación principal fijada antes de la prueba
PRINCIPAL = {"modelo_a": "qwen2.5-7b", "modelo_b": "qwen3.5-9b", "metodo": "N1", "orientacion": "senales_en_filas"}

## 3. Carga

Se avisa si alguna ejecución está incompleta (menos de 4 variantes × ventanas): puede ser una que se cortó y no se reanudó.

In [ ]:
# --- Ventanas con tu referencia y ejecuciones de cada modelo ---
VENTANAS = exp2.cargar_ventanas(f"{RUTA_ENTRADAS}/{'ventanas_desarrollo' if CONJUNTO == 'dev' else 'ventanas_prueba'}.csv")
ejecuciones = EJECUCIONES or exp2.ultimas_ejecuciones(RUTA_EJECUCIONES, CONJUNTO)

partes = [exp2.cargar(REFERENCIA_QWEN25[CONJUNTO], "qwen2.5-7b", VENTANAS)]
for modelo, base in ejecuciones.items():
    partes.append(exp2.cargar(f"{RUTA_EJECUCIONES}/{base}.csv", modelo, VENTANAS))
    esperadas = 4 * len(VENTANAS)
    aviso = "" if len(partes[-1]) == esperadas else f"  AVISO: incompleta ({len(partes[-1])} de {esperadas} llamadas)"
    print(f"{modelo}: {base}{aviso}")
df = pd.concat(partes, ignore_index=True)
MODELOS = list(dict.fromkeys(df["modelo"]))
n_eval = sum(exp2.MAPEO_REFERENCIA.get(w["referencia"]) is not None for w in VENTANAS)
n_anom = sum(w["referencia"] == "anomalia" for w in VENTANAS)
print(f"{CONJUNTO}: {len(VENTANAS)} ventanas ({n_eval} sin dudosas) | modelos: {MODELOS}")
print(f"Referencia trivial «siempre anomalía»: {n_anom}/{n_eval} ({n_anom / n_eval:.1%})")

## 4. Resultados

In [ ]:
# --- 4.1 Comparación principal (fijada antes de la prueba) ---
tabla_principal = exp2.comparacion_principal(df, **PRINCIPAL)
display(tabla_principal)
if CONJUNTO != "test":
    print("Nota: en desarrollo es solo orientativa; la comparación principal se decide en la prueba.")

In [ ]:
# --- 4.2 Métricas por modelo y variante, y acierto de un vistazo (modelos en filas, variantes en columnas) ---
tabla_metricas = exp2.metricas(df)
display(tabla_metricas)
tabla_acierto = (tabla_metricas.assign(variante=tabla_metricas["metodo"] + " | " +
                                       tabla_metricas["orientacion"].str.replace("_en_filas", ""))
                 .pivot(index="modelo", columns="variante", values="aciertos").loc[MODELOS])
display(tabla_acierto)

In [ ]:
# --- 4.3 Acierto por grupo de ventanas (con alerta, alta actividad sin alerta, al azar) ---
tabla_grupos = exp2.metricas_por_grupo(df)
display(tabla_grupos.pivot_table(index=["modelo", "metodo", "orientacion"], columns="grupo",
                                 values="aciertos", aggfunc="first"))

In [ ]:
# --- 4.4 N0 frente a N1 en cada modelo, cada par de modelos en cada variante y efecto de las dudosas ---
tabla_n0_n1 = exp2.n0_frente_a_n1(df)
tabla_pares = exp2.todos_pareados(df, MODELOS)
tabla_dudosas = exp2.dudosas(df)
display(tabla_n0_n1, tabla_pares, tabla_dudosas)

In [ ]:
# --- 4.5 Por ventana: referencia y veredicto de cada modelo y variante (1 = anomalía, · = sin veredicto) ---
df["columna"] = df["modelo"] + " | " + df["metodo"] + " | " + df["orientacion"].str.replace("_en_filas", "")
por_ventana = (df.assign(celda=df["veredicto"].map(lambda v: "·" if pd.isna(v) else str(int(v))))
                 .pivot(index="id", columns="columna", values="celda"))
por_ventana = df.drop_duplicates("id").set_index("id")[["grupo", "referencia"]].join(por_ventana)
with pd.option_context("display.max_rows", 100, "display.max_columns", 50):
    display(por_ventana)

## 5. Revisión de las respuestas

- **5.1:** una ventana: la gráfica con su tabla (nombres cortos) y el veredicto y la probabilidad de cada modelo y variante.
- **5.2:** el informe HTML con todas las ventanas y **las respuestas completas del modelo**. Lleva los nombres reales: se guarda en `privado/experimento_2/informes/`. Descárgalo de Drive y ábrelo en el navegador (funciona sin conexión).

In [ ]:
# --- 5.1 Una ventana: gráfica + tabla y el veredicto de cada modelo y variante ---
import plotly.io as pio
from visualizacion import graficar_ventana_y_tabla

pio.renderers.default = "vscode"   # "colab" si lo ejecutas en Colab desde el navegador
ID_VENTANA = VENTANAS[0]["id"]     # cambia el identificador para ver otra

w = next(v for v in VENTANAS if v["id"] == ID_VENTANA)
graficar_ventana_y_tabla(fuente.ventana(w["conjunto"], w["indice"]), NOMBRES_CONTADORES,
                         f"{ID_VENTANA} · referencia: {w['referencia']} · grupo: {w['grupo']}")
display(exp2.detalle_ventana(df, ID_VENTANA)[["modelo", "metodo", "orientacion", "veredicto", "p_interna"]])

In [ ]:
# --- 5.2 Informe HTML con las respuestas del modelo (privado) ---
correspondencia = leer_correspondencia()
nombres_reales = [correspondencia[n] for n in NOMBRES_CONTADORES]   # los que vio el modelo
os.makedirs(RUTA_INFORMES, exist_ok=True)
valores = {w["id"]: fuente.ventana(w["conjunto"], w["indice"]) for w in VENTANAS}
ruta_informe = exp2.informe_html(df, valores, nombres_reales,
                                 f"{RUTA_INFORMES}/informe_{CONJUNTO}_{datetime.now():%Y%m%d_%H%M}.html",
                                 f"Experimento 2 · {CONJUNTO} · respuestas por ventana")
print("Informe (privado):", ruta_informe)

## 6. Guardado

En `resultados/experimento_2/<conjunto>/`: las tablas en CSV y un JSON con todo lo anterior y una fila por llamada (sin textos del modelo). Es lo que se puede compartir.

In [ ]:
# --- 6. Cifras (sin textos del modelo) en resultados/experimento_2 ---
carpeta = f"{RUTA_EXPERIMENTO_2}/{CONJUNTO}"
os.makedirs(carpeta, exist_ok=True)
tablas = {"comparacion_principal": tabla_principal, "metricas": tabla_metricas, "grupos": tabla_grupos,
          "n0_frente_a_n1": tabla_n0_n1, "pares_de_modelos": tabla_pares, "dudosas": tabla_dudosas}
for nombre, tabla in tablas.items():
    tabla.to_csv(f"{carpeta}/{nombre}.csv", index=False)

registros = lambda t: json.loads(t.to_json(orient="records"))   # NaN -> null y tipos de numpy -> JSON
COLUMNAS = ["modelo", "id", "metodo", "orientacion", "veredicto", "p_interna", "json_valido", "cortada",
            "referencia", "grupo"]
resumen = {"conjunto": CONJUNTO, "fecha": datetime.now().strftime("%Y-%m-%d %H:%M"),
           "ejecuciones": {"qwen2.5-7b": os.path.basename(REFERENCIA_QWEN25[CONJUNTO])[:-4], **ejecuciones},
           "comparacion_principal_fijada": PRINCIPAL, **{k: registros(t) for k, t in tablas.items()},
           "llamadas": registros(df[COLUMNAS])}
ruta = f"{carpeta}/comparacion_{CONJUNTO}_{datetime.now():%Y%m%d_%H%M}.json"
with open(ruta, "w", encoding="utf-8") as f:
    json.dump(resumen, f, ensure_ascii=False, indent=1)
print("Guardado en:", carpeta, "| resumen para compartir:", os.path.basename(ruta))